# Inference from a tabular model

Run Setup and Load the model before presenting. The demonstrations follow the slides:
First prediction → Prepare the same record → Try another patient.


## Setup — before class

Use a Jupyter kernel with PyTorch and pandas installed. These files run offline;
model weights and saved preprocessing are already included. Setup locates the
checkout, reads the demo CSV and makes our `heart_model` helper importable.
The next cell loads the model. See `models/heart/README.md`.


In [ ]:
from pathlib import Path
import csv
import sys

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "demos/heart_model.py").is_file()
)
sys.path.insert(0, str(ROOT / "demos"))

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from heart_model import prepare

torch.manual_seed(497030)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
with (ROOT / "data/heart/demo.csv").open(newline="") as file:
    records = list(csv.DictReader(file))
first = records[0]
pain_names = {
    1: "Typical angina", 2: "Atypical angina",
    3: "Non-anginal pain", 4: "Asymptomatic",
}

def patient_table(record):
    return pd.Series({
        "Age (years)": float(record["age"]),
        "Resting blood pressure (mm Hg)": float(record["trestbps"]),
        "Cholesterol (mg/dL)": float(record["chol"]),
        "Maximum heart rate (beats/min)": float(record["thalach"]),
        "Chest-pain category": pain_names[int(record["cp"])],
    }, name="Patient record")


## Load the model

This course-trained network has eight inputs, one hidden layer with 12 units
and ReLU, and two output scores (absence, presence). It was trained on 242
labeled UCI Cleveland records; the 61 demo records were kept out of training.

Our `tools/train_heart_model.py` saved two files in `models/heart/`: `weights.pt`
holds learned weights and biases; `metadata.json` records feature order,
training means and standard deviations, and class labels. Our `load_model()`
helper rebuilds the network, restores the weights, and reads that JSON into
the Python dictionary `metadata`. The slide code uses these returned objects.

Start with the [model README](../models/heart/README.md) for input units,
normalization steps, category encoding, and output interpretation. Metadata
holds the values used by our code; the README explains how to use them.
The supplied `prepare(records, metadata)` helper implements input preparation.


In [ ]:
from heart_model import load_model

model, metadata = load_model()

print(metadata["continuous"])
# ['age', 'trestbps', 'chol', 'thalach']
print(metadata["labels"])
# ['absence', 'presence']


## First prediction

Can these measurements predict whether disease was recorded? The model's
classes are 0 = absence and 1 = presence. Keep the recorded label out of the input.


In [ ]:
display(patient_table(first))


In [ ]:
# The helper prepares this first input. We unpack its work next.
first_x = prepare([first], metadata)
with torch.inference_mode():
    first_scores = model(first_x)
    first_probabilities = first_scores.softmax(dim=1)

first_class = first_probabilities.argmax(dim=1).item()
print("Prediction:", metadata["labels"][first_class])
display(pd.Series(
    first_probabilities[0].tolist(), index=metadata["labels"], name="Class score"
).round(3))


Pause for **The first prediction** in the slides. We have a result;
now inspect what the model received and how it produced that label.


## Prepare the same record

Before training, the preparation script computed a mean and population
standard deviation (`correction=0`) for each measurement using only the 242
training records. It saved them in `metadata.json` and standardized each
training measurement with `(value - mean) / std`.

Reuse those saved means and standard deviations for each numerical
feature. Do not compute new statistics from this one patient or the demo batch.
The chest-pain category uses four one-hot positions in source-code order 1–4;
these 0/1 indicators are not standardized.


In [ ]:
numeric = torch.tensor([[67., 160., 286., 108.]])
mean = torch.tensor(metadata["mean"])
std = torch.tensor(metadata["std"])
standardized = (numeric - mean) / std

display(pd.DataFrame({
    "Original": numeric[0].tolist(),
    "Saved mean": mean.tolist(),
    "Saved std": std.tolist(),
    "Standardized": standardized[0].tolist(),
}, index=metadata["continuous"]).round(3))


In [ ]:
cp = torch.tensor([4])
pain = F.one_hot(cp - 1, num_classes=4).float()
print("Chest-pain indicators:", pain)

x = torch.cat([standardized, pain], dim=1)
print("Complete input:", x)
print("Shape:", x.shape)
display(pd.Series(
    x[0].tolist(),
    index=[*metadata["continuous"], "cp=1", "cp=2", "cp=3", "cp=4"],
    name="Model input",
).round(3))
print("Matches the first prepared input:", torch.allclose(x, first_x))


## Run the model

One input row is one patient. Evaluation behavior and inference mode are
separate settings: choose prediction behavior and skip bookkeeping for training.


In [ ]:
model.eval()
with torch.inference_mode():
    scores = model(x)
print("Output shape:", scores.shape)
display(pd.Series(
    scores[0].tolist(), index=metadata["labels"], name="Raw score"
).round(3))


## From scores to a label

Softmax makes two nonnegative scores that sum to one. Argmax selects the winning
column's index; the model's label list gives that index its meaning.
This transformation is different from mean/std standardization.


In [ ]:
probabilities = scores.softmax(dim=1)
class_id = probabilities.argmax(dim=1).item()
prediction = metadata["labels"][class_id]

display(pd.DataFrame({
    "Raw score": scores[0].tolist(),
    "After softmax": probabilities[0].tolist(),
}, index=metadata["labels"]).round(3))
print("Winning index:", class_id, "Prediction:", prediction)


## Try another patient

Inspect the next record before predicting. Then compare the predicted label
with the recorded diagnosis, which is not passed into the model.


In [ ]:
second = records[1]
display(patient_table(second))


In [ ]:
second_x = prepare([second], metadata)
with torch.inference_mode():
    second_scores = model(second_x)
    second_probabilities = second_scores.softmax(dim=1)
second_class = second_probabilities.argmax(dim=1).item()
print("Prediction:", metadata["labels"][second_class])
print("Recorded label:", metadata["labels"][int(second["target"])])
print("Presence score:", round(second_probabilities[0, 1].item(), 3))


## Several patients

Use the first three demo records in file order. Each row still means one patient.


In [ ]:
batch_x = prepare(records[:3], metadata)
with torch.inference_mode():
    batch_scores = model(batch_x)
    batch_probabilities = batch_scores.softmax(dim=1)
batch_classes = batch_probabilities.argmax(dim=1)

print("Input:", batch_x.shape, "Output:", batch_scores.shape)
comparison = pd.DataFrame({
    "Recorded label": [metadata["labels"][int(r["target"])] for r in records[:3]],
    "Prediction": [metadata["labels"][i] for i in batch_classes.tolist()],
    "Presence score": batch_probabilities[:, 1].tolist(),
}, index=[1, 2, 3])
comparison.index.name = "Demo record"
display(comparison.round(3))


The second prediction disagrees with the recorded label despite a large score.
These scores are not calibrated clinical probabilities. This small classroom
model is not a clinically validated system; its preparation and evaluation notes
are in `models/heart/README.md`.


## Optional: all demo records

All 61 demo records were excluded from model preparation. This is an instructor
check, not a new lesson on data splitting or a clinical performance claim.


In [ ]:
all_x = prepare(records, metadata)
with torch.inference_mode():
    all_scores = model(all_x)
all_classes = all_scores.argmax(dim=1)
recorded = torch.tensor([int(row["target"]) for row in records])
print("Correct:", int((all_classes == recorded).sum()), "/", len(records))
